<a href="https://colab.research.google.com/github/JoviWZhu/20206RAG/blob/RAG-Bot/FineWeb_Edu_RAG_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets sentence-transformers faiss-cpu groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.2 MB/s eta 0:00:00


In [3]:
!pip install datasets sentence-transformers faiss-cpu huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.2 MB/s eta 0:00:00


In [4]:
import os
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from huggingface_hub import InferenceClient


In [6]:

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
# Replace with your free Hugging Face Read Token

from google.colab import userdata
HF_TOKEN_DEV = userdata.get('HF_TOKEN_DEV')

client = InferenceClient(token=HF_TOKEN_DEV)

print("⚡ Step 1: Downloading a sample of FineWeb-Edu from Hugging Face...")
# Loading a small streaming sample to keep memory usage low in Colab
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
dataset_head = dataset.take(100)



documents = [doc["text"] for doc in dataset_head]
print(f"✅ Successfully loaded {len(documents)} educational documents.")

⚡ Step 1: Downloading a sample of FineWeb-Edu from Hugging Face...


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

✅ Successfully loaded 100 educational documents.


In [13]:
# UPDATED STEP 2: USE INDEXFLATIP & NORMALIZATION
# ==========================================
print("\n⚡ Step 2: Initializing the embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("⚡ Indexing documents into FAISS Local Vector Database using Cosine Similarity...")
# 1. Generate text embeddings normally
document_embeddings = embedding_model.encode(documents, convert_to_numpy=True)

# 2. CRITICAL STEP: Normalize the text vectors so they have a length of 1
faiss.normalize_L2(document_embeddings)

# 3. Create an Inner Product Index instead of FlatL2
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(document_embeddings)
print("✅ Local Vector DB is built using Cosine Similarity.")


⚡ Step 2: Initializing the embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⚡ Indexing documents into FAISS Local Vector Database using Cosine Similarity...
✅ Local Vector DB is built using Cosine Similarity.


In [29]:
# ==========================================
# 3 & 4. FIXED RAG EXECUTION FUNCTION WITH ALL VARIABLES
# ==========================================
def run_rag_query(user_question, top_k=2):
    print(f"\n🔍 User Question: '{user_question}'")
    print(f"⚡ Step 3: Searching local database for relevant FineWeb-Edu facts...")

    # 3a. Vectorize user query
    question_embedding = embedding_model.encode([user_question], convert_to_numpy=True)

    # Normalize for Cosine Similarity metric
    faiss.normalize_L2(question_embedding)

    # 3b. Search local index
    distances, indices = index.search(question_embedding, top_k)

    # 3c. Extract text matching indices (flattening matrix output using indices[0])
    retrieved_contexts = [documents[idx] for idx in indices[0]]
    print("✅ Facts retrieved successfully.")

    # 3d. PACKAGE IT UP: Construct strict prompt instruction
    context_str = "\n---\n".join(retrieved_contexts)

    # CRITICAL FIX: Defining system_prompt clearly before loading it below
    system_prompt = (
        "You are an academic expert assistant. Answer the user's question using ONLY the provided text context from FineWeb-Edu. "
        "If the answer cannot be confidently derived from the context, reply with 'I cannot find the answer in the provided documents.' "
        "Do not make up facts or use outside knowledge."
    )

    user_prompt = f"Context from FineWeb-Edu:\n{context_str}\n\nQuestion: {user_question}\nAnswer:"

    # ==========================================
    # 4. EXTERNAL API GENERATION STEP (HUGGING FACE)
    # ==========================================
    print("⚡ Step 4: Sending the packaged prompt bundle to Hugging Face Serverless API...")

    try:
        # Using the direct client.chat_completion endpoint
        response = client.chat_completion(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=500,
            temperature=0.2
        )

        print("\n🤖 [Llama 3.1 RAG Response]:")

        # Completely bulletproof type checking framework for choices output
        if hasattr(response, 'choices') and len(response.choices) > 0:
            choice = response.choices[0]
            if hasattr(choice, 'message'):
                print(choice.message.content)
            elif isinstance(choice, dict) and 'message' in choice:
                print(choice['message']['content'])
            else:
                print(choice)
        else:
            print(response)

    except Exception as e:
        print(f"❌ Error communicating with Hugging Face API: {e}")

# ==========================================
# 5. RUN THE QUERY OVER RE-INITIALIZED BLOCKS
# ==========================================
run_rag_query("What are the key concepts of physical science or historical events explained here?", top_k=2)



🔍 User Question: 'What are the key concepts of physical science or historical events explained here?'
⚡ Step 3: Searching local database for relevant FineWeb-Edu facts...
✅ Facts retrieved successfully.
⚡ Step 4: Sending the packaged prompt bundle to Hugging Face Serverless API...

🤖 [Llama 3.1 RAG Response]:
I cannot find the answer in the provided documents. The text appears to be about astrology and the book "Astrological Dignities" by Lee Lehman, and does not discuss physical science or historical events.


In [30]:
# Ask a question that matches the text it actually has in its local index
run_rag_query("What book or author is being discussed in these documents?", top_k=2)



🔍 User Question: 'What book or author is being discussed in these documents?'
⚡ Step 3: Searching local database for relevant FineWeb-Edu facts...
✅ Facts retrieved successfully.
⚡ Step 4: Sending the packaged prompt bundle to Hugging Face Serverless API...

🤖 [Llama 3.1 RAG Response]:
The book being discussed is "The Glory Field" by Lee Lehman, and the author being discussed is Lee Lehman.


In [32]:
# Ask a question that matches the text it actually has in its local index
run_rag_query("One long sentence in these documents?", top_k=2)



🔍 User Question: 'One long sentence in these documents?'
⚡ Step 3: Searching local database for relevant FineWeb-Edu facts...
✅ Facts retrieved successfully.
⚡ Step 4: Sending the packaged prompt bundle to Hugging Face Serverless API...

🤖 [Llama 3.1 RAG Response]:
The sentence "The rock rolled off the table, landed on top of a skateboard, and proceeded to roll down the hill until it was stopped by a wall." is an example of a long sentence in the provided documents.
